In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from tqdm.notebook import tqdm
pd.set_option('Display.max_columns', None)

In [2]:
import pandas as pd
import numpy as np
import re
from matplotlib import pyplot as plt
#from tqdm.notebook import tqdm
pd.set_option('Display.max_columns', None)

import sys
sys.path.append('../../ss_lal_military/src')
sys.path.append('../src/')
sys.path.append('../migrant/notebooks/utilities/')

In [3]:
from subprocess import Popen, PIPE, run
from getpass import getpass

def kinit(username: str):
    """
    Obtain and cache an initial ticket-granting ticket for principal.

    :param username: principal username.
    :return:
    """
    kinit_path = '/usr/bin/kinit'
    kinit_args = [kinit_path, '%s' % username]
    pipe = Popen(kinit_args, stdin=PIPE, stdout=PIPE, stderr=PIPE)
    pipe.communicate(input='{}\n'.format(getpass()).encode('utf-8'))
    klist_result = run(['klist'], stdout=PIPE).stdout.decode('utf-8')
    print(klist_result)
    
kinit("21417984_omega-sbrf-ru@DF.SBRF.RU")

 ··············


Ticket cache: FILE:/tmp/krb5cc_1623419065
Default principal: 21417984_omega-sbrf-ru@DF.SBRF.RU

Valid starting       Expires              Service principal
09/25/2025 10:38:18  09/26/2025 10:36:59  krbtgt/DF.SBRF.RU@DF.SBRF.RU
	renew until 10/02/2025 10:38:18



In [4]:
#spark
from spark_utils import *

spark = get_spark_session('platon_target_agent')

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/09/25 10:38:23 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
25/09/25 10:38:23 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
25/09/25 10:38:23 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.
25/09/25 10:38:23 WARN Utils: Service 'SparkUI' could not bind on port 4043. Attempting port 4044.
25/09/25 10:38:28 WARN Client: Exception encountered while connecting to the server 
org.apache.hadoop.ipc.RemoteException(org.apache.hadoop.ipc.StandbyException): Operation category READ is not supported in state standby. Visit https://s.apache.org/sbnn-error
	at org.apache.hadoop.security.SaslRpcClient.saslConnect(SaslRpcClient.java:376)
	at org.apache.hadoop.ipc.Client$Connection.setupSaslConnection(Client.java:623)
	at org.apache.hadoop.ipc.Client$Connection.access$

In [5]:
spark

In [ ]:
MVSC_2032_BANNER_SALE_PL_NSP_2_0226_SCSELFS_9856
MVSC_2032_BANNER_SALE_PL_NSP1_0226_SCSELFS_99856

In [9]:
company = {
# I can not show name of companies, because i am under NDA
}

In [10]:
company_id = ['114177','114392','114391','114381','114389','114383','114388','114386','114376','114369','114364','114372','114356','114357','114375','114368','114367','114102','114126','114047','114177','113704','114102','114126','114047','114177','113704','114102','112711','112713','113245','112698','112714','112715','113386','112457','113528','112448','112455','112716','112708','112710','112454','112696','113704','112695','112691','113585','112678','112697','112690','113519','113491','112694','112693','113242','112014','112073','112075','112095','112100','112457','112465','112455','112454','112071','112220','112176','112085','112617','112089','112218','112012','112072','112079','112078','112088','112205','111007','111015','111010','111025','111019','111041','111021','111298','110774','111164','111014','111046','111042','111009','111004','111510','111699','111011','111007','110418','111015','111010','110417','110422','110383','111019','110421','111041','110518','110774','111014','110500','111046','110519','111042','110403','111004','110503','110947','110410','110502','110347','110382','110412','110406','111011',
    '110405',
    '109555',
    '109522',
    '109754',
    '109571',
    '109502',
    '109541',
    '109597',
    '109503',
    '109755',
    '109528',
    '109524',
    '109501',
    '108895',
    '108667',
    '107947',
    '108326',
    '108138',
    '107774',
    '107055',
    '106949',
    '107024'
]

In [11]:
company_name = [
# I can not show name of companies names, because i am under NDA
]

In [ ]:
s_grnplm_ld_rozn_electron_aaas_dm.evk_hist = prx_bpm_kompanii_custom_rb_evk.dm_union_campaign_history
s_grnplm_ld_rozn_electron_aaas_dm.evk_dic = prx_bpm_kompanii_custom_rb_evk.ref_union_campaign_dic_hdp

In [3]:
"""Функции для взаимодействия с Greenplum."""

import getpass
import pandas as pd
import sqlalchemy
from sqlalchemy import text


def get_sqlalchemy_engine():
    parameters = {
        "user": getpass.getuser().split("_")[0],
        "host": "gp_dns_pkap1150.gp.df.sbrf.ru",
        "port": 5432,
        "database": "gp_rozn2",
    }

    url = "postgresql+psycopg2://{user}@{host}:{port}/{database}".format(**parameters)
    engine = sqlalchemy.create_engine(url=url)

    return engine

class greenplum_con():
    """GP подключение"""
    def __init__(self):
        self.connection = get_sqlalchemy_engine().connect()
        
    def take(self, query):
        try:
            data = pd.read_sql_query(text(query), self.connection)
            return data
        except Exception as e:
            self.close()
            self.connect()
            print(str(e))
    
    def connect(self):
        self.connection = get_sqlalchemy_engine().connect()
        
    def close(self):
        self.connection.close()

def print_execution_plan(query: str, engine) -> None:
    """Напечатать план выполнения запроса.

    Перед запуском потенциально тяжелых запросов рекомендуется проверять план
    выполнения. Стоимость не должна превышать "сотни тысяч - миллионы".

    """

    plan = engine.execute(f"EXPLAIN {query}").fetchall()
    plan = "\n".join(map(lambda x: x[0], plan))
    print(plan)

In [4]:
gp = greenplum_con()

In [10]:
gp.close()

In [5]:
engine = get_sqlalchemy_engine()
con = engine.connect()

In [12]:
con.rollback()

In [12]:
df_zero = pd.read_sql(text(f'''
select * from s_grnplm_ld_rozn_electron_ss_users_temp.bpm_premier_company_zero
'''), con)

In [18]:
df_zero.to_parquet('target_company_zero.parquet')

In [19]:
df_one = pd.read_sql(text(f'''
select * from s_grnplm_ld_rozn_electron_ss_users_temp.bpm_premier_company_one
'''), con)

In [23]:
df_one.to_parquet('target_company_one.parquet')

In [25]:
df = pd.concat([df_zero, df_one], axis=0)

In [28]:
df.to_parquet('target.parquet')

In [6]:
campaigns = [
# I can not show name of companies names, because i am under NDA
]

In [7]:
date_range = pd.date_range(start = '2025-01-01', end = '2025-09-30', freq = "MS").strftime("%Y-%m-%d").tolist()

In [8]:
date_range

['2025-01-01',
 '2025-02-01',
 '2025-03-01',
 '2025-04-01',
 '2025-05-01',
 '2025-06-01',
 '2025-07-01',
 '2025-08-01',
 '2025-09-01']

In [21]:
con.rollback()

In [9]:
df_list = []
for dt in date_range:
    
    lam = pd.read_sql(text(f'''
    select campaign_name, date('{dt}') as campaign_start_dt, unique_only_nflag, count(distinct a.epk_id) as clients_cnt
    from s_grnplm_ld_rozn_electron_aaas_dm.evk_hist a 
    join s_grnplm_ld_rozn_electron_aaas_dm.evk_dic b on a.sk_id = b.sk_id
    left join s_grnplm_ld_rozn_electron_aaas_dm.gl_evo_resp_cln_month c on a.epk_id = c.epk_id and b.ab_test_id = c.ab_test_id and resp_month >= '2025-05-01'
    where campaign_name in ({','.join([f"'{w}'" for w in campaigns])}) and a.start_dt >= date('{dt}') and a.end_dt <= last_day(date('{dt}') + interval '1 month') 
    and b.start_dt >= date('{dt}') and b.end_dt <= last_day(date('{dt}') + interval '1 month') and unique_only_nflag is not null
    group by campaign_name, date('{dt}'), unique_only_nflag
    '''), con)
    df_list.append(lam)
    print(dt, ': done')

2025-01-01 : done
2025-02-01 : done
2025-03-01 : done
2025-04-01 : done
2025-05-01 : done
2025-06-01 : done
2025-07-01 : done
2025-08-01 : done
2025-09-01 : done


In [10]:
df_monthly = pd.concat(df_list, ignore_index=True)

In [11]:
df_monthly_new = df_monthly.copy()

In [12]:
df_monthly_new.to_parquet('df_group_by_new.parquet')

In [17]:
df_monthly[(df_monthly['unique_only_nflag'] == 1)]['clients_cnt'].sum()

22692

In [22]:
df_monthly.to_parquet('df_group_by.parquet')

In [23]:
df_monthly['campaign_name'].nunique()

11

In [6]:
campgains_new =[
# I can not show name of companies names, because i am under NDA
]

In [7]:
df_new_target_0 = pd.read_sql(text(f'''
select a.epk_id, a.start_dt, a.end_dt, unique_only_nflag, campaign_name
    from s_grnplm_ld_rozn_electron_aaas_dm.evk_hist a 
    join s_grnplm_ld_rozn_electron_aaas_dm.evk_dic b on a.sk_id = b.sk_id
    left join s_grnplm_ld_rozn_electron_aaas_dm.gl_evo_resp_cln_month c on a.epk_id = c.epk_id and b.ab_test_id = c.ab_test_id and resp_month >= '2025-07-01'
    where campaign_name in ({','.join([f"'{w}'" for w in campgains_new])}) and a.start_dt >= '2025-07-01' and a.end_dt <= '2025-09-30' 
    and b.start_dt >= '2025-07-01' and b.end_dt <= '2025-09-30' and unique_only_nflag = 0
    order by random()
    limit 400000
'''), con )

In [9]:
df_new_target_0.to_parquet('target_0_new.parquet')

In [4]:
df = pd.read_parquet('../data/target.parquet')

In [6]:
df['start_dt'].unique()

array([datetime.date(2025, 6, 1), datetime.date(2025, 8, 1),
       datetime.date(2025, 5, 1), datetime.date(2025, 7, 1)], dtype=object)